# Specialiserede Modeller — 5: RNN & LSTM — modeller med hukommelse

Tabeller, billeder... nu til den tredje store datatype: **sekvenser**. Temperaturer dag
for dag, aktiekurser, hjerterytmer, sætninger — data hvor **rækkefølgen ER
informationen**. Byt om på to dage i en vejrserie, og den er ødelagt; byt om på to ord
i en sætning, og "hunden bed manden" bliver til noget helt tredje.

Dagens data: **11 års daglige temperaturer** fra Szeged i Ungarn. Missionen: forudsig
morgendagens temperatur. Og dagens vigtigste modstander er ikke en model — det er
verdens mest undervurderede prognose: *"i morgen bliver som i dag"*.

> Denne notebook er selvkørende og kræver kun viden fra Intro-ML — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå: opgaver mærket **(ekstra)** er til de hurtige, og opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes.

## Setup

In [ ]:
# Henter daglige temperaturer + fugt fra GitHub (Plan B: upload vejr_dagligt.csv.gz via mappeikonet)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/vejr_dagligt.csv.gz

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

dagligt_df = pd.read_csv("vejr_dagligt.csv.gz")
temperatur = dagligt_df["temperatur"].values.astype("float32")
print("antal dage:", len(temperatur))
dagligt_df.head(3)

Datasættet indeholder oprindeligt én måling pr. TIME i 11 år. Vi har på forhånd omregnet det
til **daglige middelværdier** (med pandas' `resample("D")`), så vi slipper for at hente den
store timefil hver gang. Resultatet: én temperatur og én luftfugtighed pr. dag.

> **Plan B:** Henter `wget`-cellen ikke filen, så upload `vejr_dagligt.csv.gz` manuelt via
> mappeikonet i Colab (filen ligger i `28-Data/MLData` på GitHub).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(temperatur)
axes[0].set_title("11 års daglige temperaturer — kan I se årstiderne?")
axes[1].plot(temperatur[365:730])
axes[1].set_title("zoom: ét enkelt år")
axes[1].set_xlabel("dag")
for axis in axes:
    axis.set_ylabel("°C")
plt.tight_layout()
plt.show()

# 1: Sekvenser — og modstanderen, der skal slås

## Første regel om tidsserier: splittet er KRONOLOGISK

I al tidligere ML blandede vi data og splittede tilfældigt. **Det er FORBUDT her.**
Modellen skal forudsige FREMTIDEN ud fra FORTIDEN — så testsættet skal ligge EFTER
træningssættet i tid. Blander man, træner modellen på tirsdag og onsdag og "forudsiger"
mandagen imellem dem — spådom med facitliste. Vi bruger de første 80 % af dagene til
træning og de sidste 20 % til test.

## Modstanderen: den naive prognose

Før man bygger NOGEN prognosemodel, skal man kende den naive baseline:
**"i morgen = i dag"**. Den kræver nul træning, nul parametre — og den er pinligt
svær at slå på temperaturer:

In [ ]:
n_train = int(0.8 * len(temperatur))

i_gaar = temperatur[n_train:-1]        # dagens temperatur...
i_morgen = temperatur[n_train + 1:]    # ...og dagen efter

naive_mae = np.abs(i_gaar - i_morgen).mean()
print(f"naiv prognose ('i morgen = i dag'): {naive_mae:.2f} graders fejl i snit")

**~1,5 grader i gennemsnitsfejl — uden at løfte en finger.** Det tal skal enhver
model måles imod. En prognosemodel, der er dårligere end "i morgen = i dag", er ikke en
prognosemodel; den er spild af strøm. (Journalist-huskeregel: når nogen praler af en
"87 % præcis AI-prognose" — spørg altid, hvad den naive baseline var!)

## Glidende vinduer: fra serie til træningsdata

Hvordan træner man overhovedet en model på én lang serie? Man klipper den op i
**vinduer**: de sidste 7 dage er input, dag nummer 8 er facit. Så glider man én dag
frem og klipper igen:

In [ ]:
window = 7
X_liste, y_liste = [], []
for i in range(len(temperatur) - window):
    X_liste.append(temperatur[i:i + window])       # 7 dage ind...
    y_liste.append(temperatur[i + window])         # ...dag 8 er facit

X_alle = np.array(X_liste)
y_alle = np.array(y_liste)
print("vinduer:", X_alle.shape, "| facit:", y_alle.shape)
print("eksempel — input:", X_alle[0].round(1), "→ facit:", round(float(y_alle[0]), 1))

In [ ]:
n_train = int(0.8 * len(X_alle))                   # kronologisk split!
X_train_np, X_test_np = X_alle[:n_train], X_alle[n_train:]
y_train_np, y_test_np = y_alle[:n_train], y_alle[n_train:]

# standardisér — med TRÆNINGSSÆTTETS gennemsnit/spredning (testen er "fremtiden"!)
gns, std = X_train_np.mean(), X_train_np.std()

X_train = torch.tensor((X_train_np - gns) / std)
X_test = torch.tensor((X_test_np - gns) / std)
y_train = torch.tensor((y_train_np - gns) / std)
y_test_celsius = torch.tensor(y_test_np)           # facit beholdes i rigtige grader

print("træning:", X_train.shape, "| test:", X_test.shape)

## Første udfordrer: et dense-netværk på vinduerne

7 tal ind, 1 tal ud — det er jo bare et almindeligt netværk fra Intro-ML! Bemærk
hvordan vi regner fejlen om til GRADER til sidst (ganger spredningen på igen), så
tallet kan sammenlignes med den naive baseline:

In [ ]:
torch.manual_seed(42)
dense_model = nn.Sequential(nn.Linear(window, 32), nn.ReLU(), nn.Linear(32, 1))
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(dense_model.parameters(), lr=0.001)

for epoch in range(1000):
    optimizer.zero_grad()
    loss = loss_fn(dense_model(X_train).squeeze(), y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    predicted = dense_model(X_test).squeeze() * std + gns    # tilbage til °C
dense_mae = (predicted - y_test_celsius).abs().mean()
print(f"dense-net: {dense_mae.item():.2f} graders fejl  (naiv: {naive_mae:.2f})")

### Opgaver

##### Opgave 1.1
Zoom ind på ét år (plottet ovenfor): cirka hvor mange grader svinger temperaturen fra
dag til dag — og hvor mange grader svinger den over et halvt år? Hvilken af de to
"bevægelser" er nemmest at forudsige, og hvorfor?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.2
Forklar med jeres egne ord, hvorfor tidsserier SKAL splittes kronologisk. Hvad ville
en model helt konkret kunne "snyde" med, hvis testdagene lå tilfældigt spredt mellem
træningsdagene?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.3
Udfyld beregningen af den naive baseline — men nu målt på PRÆCIS de samme testvinduer
som modellerne (sidste dag i vinduet som gæt). Rammer I ~samme tal som ovenfor?

In [ ]:
naive_pred = X_test_np[:, -1]        # sidste dag i hvert vindue
naive_mae_windows = np.abs(... - ...).mean()
print(f"naiv MAE på testvinduerne: {naive_mae_windows:.2f} grader")

##### Opgave 1.4 (find fejlen)
En kammerat har "forbedret" splittet ved at blande vinduerne tilfældigt før opdelingen
— "så er testen mere repræsentativ!". Koden kører fint og giver et tal. Men tallet kan
man ikke stole på. Forklar hvorfor (hint: hvor mange af de 7 dage deler vindue nr. 100
og vindue nr. 101 — og hvor ender de to vinduer efter blandingen?).

In [ ]:
mix = np.random.default_rng(42).permutation(len(X_alle))
X_mixed, y_mixed = X_alle[mix], y_alle[mix]

X_tr_b = torch.tensor((X_mixed[:n_train] - gns) / std)
X_te_b = torch.tensor((X_mixed[n_train:] - gns) / std)
y_tr_b = torch.tensor((y_mixed[:n_train] - gns) / std)
y_te_b = torch.tensor(y_mixed[n_train:])

torch.manual_seed(42)
m_b = nn.Sequential(nn.Linear(window, 32), nn.ReLU(), nn.Linear(32, 1))
opt = torch.optim.Adam(m_b.parameters(), lr=0.001)
for epoch in range(1000):
    opt.zero_grad()
    loss = loss_fn(m_b(X_tr_b).squeeze(), y_tr_b)
    loss.backward()
    opt.step()
with torch.no_grad():
    predicted_b = m_b(X_te_b).squeeze() * std + gns
print(f"MAE med blandet split: {(predicted_b - y_te_b).abs().mean().item():.2f} — kan man stole på det tal?")

##### Opgave 1.5
Hvor mange dages hukommelse hjælper dense-nettet? Prøv `vindue` på **1, 7 og 30**
(byg vinduerne og modellen forfra hver gang — skabelonen klarer det). Hvor meget
hjælper mere fortid egentlig?

In [ ]:
def dense_attempt(window_str):
    Xl, yl = [], []
    for i in range(len(temperatur) - window_str):
        Xl.append(temperatur[i:i + window_str])
        yl.append(temperatur[i + window_str])
    Xa, ya = np.array(Xl), np.array(yl)
    n = int(0.8 * len(Xa))
    g, sp = Xa[:n].mean(), Xa[:n].std()
    Xt = torch.tensor((Xa[:n] - g) / sp)
    Xv = torch.tensor((Xa[n:] - g) / sp)
    yt = torch.tensor((ya[:n] - g) / sp)
    torch.manual_seed(42)
    model_f = nn.Sequential(nn.Linear(window_str, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.Adam(model_f.parameters(), lr=0.001)
    for epoch in range(1000):
        opt.zero_grad()
        loss = loss_fn(model_f(Xt).squeeze(), yt)
        loss.backward()
        opt.step()
    with torch.no_grad():
        predicted_f = model_f(Xv).squeeze() * sp + g
    return (predicted_f - torch.tensor(ya[n:])).abs().mean().item()

print(f"vindue 7: {dense_attempt(7):.2f} grader")   # ← prøv 1, 7, 30

##### Opgave 1.6
Udfyld vindues-byggeriet: input er dagene `i` til `i + vindue`, facit er dagen lige
efter.

In [ ]:
window_test = 14
Xl, yl = [], []
for i in range(len(temperatur) - window_test):
    Xl.append(temperatur[...])
    yl.append(temperatur[...])
print(np.array(Xl).shape, np.array(yl).shape)   # skal give (4005, 14) (4005,)

##### Opgave 1.7
Gør status: slog dense-nettet den naive baseline? Med hvor meget? Og er det, ærligt
talt, besværet værd indtil videre?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.8
Endnu en doven model: 'gæt altid årsgennemsnittet'. Udfyld, så du trækker gennemsnittet
fra det rigtige svar, og se hvor meget værre den er end naiven (~1,5 grader).

In [ ]:
mean = temperatur.mean()
dumb_mae = np.abs(y_test_np - ...).mean()   # ← træk gennemsnittet fra
print(f"'gæt altid gennemsnit' MAE: {dumb_mae:.2f} grader")

##### Opgave 1.9 (ekstra)
Hvor tit rammer den naive prognose (i går = i morgen) inden for 1 grad? Kør cellen.

In [ ]:
naive = X_test_np[:, -1]         # sidste dag i hvert vindue = "i morgen bliver som i dag"
before2_1 = (np.abs(naive - y_test_np) < 1.0).mean()
print(f"naiv prognose inden for 1 grad: {before2_1:.0%} af testdagene")

# 2: LSTM — netværket der læser én dag ad gangen

Dense-nettet ser vinduet som 7 løsrevne tal — det ved ikke, at dag 3 kommer FØR dag 4
(byt to inputs om, og det er ligeglat: det lærer bare nye vægte). En **RNN**
(*recurrent neural network*) gør noget fundamentalt andet: den læser sekvensen ét
skridt ad gangen og bærer en **hukommelse** med sig:

> læs dag 1 → opdatér hukommelsen → læs dag 2 → opdatér hukommelsen →... → gæt

**LSTM** (*Long Short-Term Memory*) er den forbedrede udgave, alle bruger: dens
hukommelsescelle har små indbyggede "porte", der lærer hvad der skal **huskes**, hvad
der skal **glemmes**, og hvad der skal **siges videre**. (Portene er selv små neurale
lag — matematikken bag springer vi over med god samvittighed.)

I PyTorch er det ét lag: `nn.LSTM`. Det spiser 3D-input `(antal, tidsskridt, features)`
— vores vinduer skal altså have en ekstra dimension på (7 dage × 1 tal pr. dag), og vi
skriver **`batch_first=True`** (ellers forventer PyTorch en anden rækkefølge — en
klassisk fælde):

In [ ]:
class WeatherLSTM(nn.Module):
    def __init__(self, memory=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=memory, batch_first=True)
        self.fc = nn.Linear(memory, 1)

    def forward(self, x):                 # x: (antal, 7 dage, 1 tal)
        memories, _ = self.lstm(x)     # hukommelsen efter HVERT tidsskridt
        sidste = memories[:, -1, :]    # vi bruger den efter SIDSTE dag
        return self.fc(sidste)

X_train_3d = X_train.unsqueeze(2)         # (3209, 7) → (3209, 7, 1)
X_test_3d = X_test.unsqueeze(2)
print("LSTM-input:", X_train_3d.shape, "  ← (antal, tidsskridt, features)")

In [ ]:
torch.manual_seed(42)
lstm_model = WeatherLSTM()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

for epoch in range(60):
    for i in range(0, len(X_train_3d), 256):
        optimizer.zero_grad()
        loss = loss_fn(lstm_model(X_train_3d[i:i + 256]).squeeze(), y_train[i:i + 256])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    predicted = lstm_model(X_test_3d).squeeze() * std + gns
lstm_mae = (predicted - y_test_celsius).abs().mean()

print(f"naiv:  {naive_mae:.2f} grader")
print(f"dense: {dense_mae.item():.2f} grader")
print(f"LSTM:  {lstm_mae.item():.2f} grader")

**Dagens ærlige facit: det er (stort set) dødt løb.** Alle tre lander omkring 1,5
grader. Er LSTM'en så til grin? Nej — men temperatur-prognose ER bare et problem, hvor
"i går + støj" næsten er hele sandheden, så der er intet dybt sekvens-mønster at hente.

LSTM'ens superkræfter viser sig, når rækkefølgen bærer RIGTIG information: melodier,
hjerterytmer, sensor-mønstre — og frem for alt **sprog**, hvor ordenes rækkefølge er
alt. Husk denne notebook, når I på campens sidste dag møder sprogmodeller: en sætning
er en sekvens, og "gæt næste ord" er PRÆCIS "gæt morgendagens temperatur" — bare med
ord.

## Prognoser flere dage frem: fejlen der vokser

Én dag frem er én ting — men hvad med 14 dage? Tricket: brug modellens eget gæt som
"nyeste dag" og gæt igen (og igen...). Se, hvad der sker:

In [ ]:
start_window = X_test_3d[0]                      # de første 7 testdage (standardiseret)
forecast = []

window_now = start_window.clone()
with torch.no_grad():
    for day in range(14):
        pred = lstm_model(window_now.unsqueeze(0)).squeeze()      # gæt næste dag
        forecast.append(pred.item() * std + gns)
        # glid vinduet: smid ældste dag ud, sæt gættet ind bagerst
        window_now = torch.cat([window_now[1:], pred.reshape(1, 1)])

plt.plot(range(14), y_test_np[:14], "o-", label="faktisk temperatur")
plt.plot(range(14), forecast, "s--", label="LSTM-prognose (gæt på gæt)")
plt.xlabel("dage ud i fremtiden")
plt.ylabel("°C")
plt.legend()
plt.title("Prognose 14 dage frem — fejlen vokser")
plt.show()

### Opgaver

##### Opgave 2.1
Udfyld omformningen fra 2D-vinduer til LSTM'ens 3D-format — og tjek shapen. (Hvilken
dimension mangler, og hvad betyder den?)

In [ ]:
X_eksempel = X_test[:5]              # (5, 7) — 5 vinduer á 7 dage
X_eksempel_3d = X_eksempel....(...)
print(X_eksempel_3d.shape)            # skal give (5, 7, 1)

##### Opgave 2.2 (find fejlen)
Denne kammerat glemte 3D-formatet og fodrer LSTM'en direkte med 2D-vinduerne. Kør
cellen, læs fejlbeskeden, og fix koden med ét kald.

In [ ]:
torch.manual_seed(42)
m_error = WeatherLSTM()
print(m_error(X_test[:5]))

##### Opgave 2.3
Prøv hukommelses-størrelser på **4, 32 og 128** og sammenlign MAE (og fornemmelsen af
træningstid). Betaler en stor hukommelse sig på det her problem?

In [ ]:
memory = 4        # ← prøv 4, 32, 128
torch.manual_seed(42)
m_h = WeatherLSTM(memory=memory)
optimizer = torch.optim.Adam(m_h.parameters(), lr=0.001)
for epoch in range(60):
    for i in range(0, len(X_train_3d), 256):
        optimizer.zero_grad()
        loss = loss_fn(m_h(X_train_3d[i:i + 256]).squeeze(), y_train[i:i + 256])
        loss.backward()
        optimizer.step()
with torch.no_grad():
    f = m_h(X_test_3d).squeeze() * std + gns
print(f"hukommelse {memory}: {(f - y_test_celsius).abs().mean().item():.2f} grader")

##### Opgave 2.4
Giv LSTM'en en måneds hukommelse: byg vinduer med `vindue = 30` (genbrug opskriften fra
afsnit 1 — husk nyt gennemsnit/spredning og kronologisk split) og træn. Hjælper de
ekstra 23 dage?

In [ ]:
window30 = 30
Xl, yl = [], []
for i in range(len(temperatur) - window30):
    Xl.append(temperatur[i:i + window30])
    yl.append(temperatur[i + window30])
Xa, ya = np.array(Xl), np.array(yl)
n = int(0.8 * len(Xa))
g30, s30 = Xa[:n].mean(), Xa[:n].std()
Xt30 = torch.tensor((Xa[:n] - g30) / s30).unsqueeze(2)
Xv30 = torch.tensor((Xa[n:] - g30) / s30).unsqueeze(2)
yt30 = torch.tensor((ya[:n] - g30) / s30)

torch.manual_seed(42)
m30 = WeatherLSTM()
optimizer = torch.optim.Adam(m30.parameters(), lr=0.001)
# ← træn (samme loop som før) og mål MAE i grader
...

##### Opgave 2.5
Kør 14-dages-prognosen igen, men start et andet sted i testsættet (fx
`X_test_3d[500]`, og sammenlign med `y_test_np[500:514]`). Hvor mange dage frem er
prognosen reelt brugbar — og hvad konvergerer den imod, når den giver op?

In [ ]:
start = 0        # ← prøv fx 500 og 900 (husk at flytte facit-linjen tilsvarende!)
window_now = X_test_3d[start].clone()
forecast = []
with torch.no_grad():
    for day in range(14):
        pred = lstm_model(window_now.unsqueeze(0)).squeeze()
        forecast.append(pred.item() * std + gns)
        window_now = torch.cat([window_now[1:], pred.reshape(1, 1)])

plt.plot(range(14), y_test_np[start:start + 14], "o-", label="faktisk")
plt.plot(range(14), forecast, "s--", label="prognose")
plt.legend()
plt.show()

##### Opgave 2.6
Hvorfor er "i morgen = i dag" så svær at slå på DAGLIGE temperaturer? Og på hvilke
slags sekvenser ville I forvente, at en LSTM til gengæld ville SMADRE den naive
baseline? Giv mindst to eksempler og begrund.

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 2.7 (ekstra)
Giv modellen flere sanser: byg vinduer med **to features pr. dag** — temperatur OG
luftfugtighed (kolonnen `fugt` i `dagligt_df`). LSTM'en skal så have
`input_size=2`. Hjælper fugtigheden på prognosen?

In [ ]:
humidity = dagligt_df["fugt"].values.astype("float32")   # luftfugtighed pr. dag (allerede resamplet)
print(len(humidity), len(temperatur))     # samme antal dage

# byg vinduer med shape (antal, 7, 2): [temperatur, fugt] pr. dag
Xl, yl = [], []
for i in range(len(temperatur) - 7):
    pair = np.stack([temperatur[i:i + 7], humidity[i:i + 7]], axis=1)   # (7, 2)
    Xl.append(pair)
    yl.append(temperatur[i + 7])
Xa, ya = np.array(Xl), np.array(yl)
print(Xa.shape)

# split + standardisér pr. feature + træn en VejrLSTM med input_size=2
...

##### Opgave 2.8
Kig nærmere på LSTM'ens fejl dag for dag. Udfyld tærsklen, og se hvor tit den rammer
inden for 1 grad.

In [ ]:
error = (predicted - y_test_celsius).abs()
print(f"bedste dag: {error.min():.2f}°, værste: {error.max():.2f}°, snit: {error.mean():.2f}°")
before2_1 = (error < ...).float().mean()      # ← tærskel på 1.0 grad
print(f"inden for 1 grad: {before2_1:.0%} af dagene")

##### Opgave 2.9 (ekstra)
Lad LSTM'en spå 30 dage frem ved at fodre sine egne gæt tilbage. Kør cellen — hvad sker
der med prognosen langt ude i fremtiden?

In [ ]:
window = X_test_3d[100].clone()
preds = []
with torch.no_grad():
    for day in range(30):
        g = lstm_model(window.unsqueeze(0)).squeeze()
        preds.append(g.item() * std + gns)
        window = torch.cat([window[1:], g.reshape(1, 1)])
print(f"dag 1: {preds[0]:.1f}°   dag 30: {preds[-1]:.1f}°")
print("kurven flader ud mod et middelniveau — usikkerheden æder detaljerne")